In [1]:
import os
import requests
import tarfile
import time
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, random_split ,Subset
import torchvision.models as models
import torch.nn as nn
import torch
import PIL.Image
import pathlib
from torchsummary import summary
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import torch.optim as optim
import pandas as pd
from torchvision.transforms import functional as TF
import timm
from transformers import AutoImageProcessor, AutoModelForImageClassification
import torchvision.transforms as transforms
import torchvision.transforms.v2 as v2
from torchvision.transforms import AutoAugment, AutoAugmentPolicy
from sklearn.metrics import classification_report
from torch.cuda.amp import autocast, GradScaler # Mixed Precision
from sklearn.metrics import classification_report
from transformers import BeitImageProcessor, BeitForImageClassification


if torch.cuda.is_available():
    device = "cuda" # Use NVIDIA GPU (if available)
elif torch.backends.mps.is_available():
    device = "mps" # Use Apple Silicon GPU (if available)
else:
    device = "cpu" # Default to CPU if no GPU is available

class_names = ['birds', 'bottles', 'breads', 'butterflies', 'cakes', 'cats', 'chickens', 'cows', 'dogs', 'ducks',
                  'elephants', 'fishes', 'handguns', 'horses', 'lions', 'lipsticks', 'seals', 'snakes', 'spiders', 'vases']


/home/myenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!nvidia-smi

Sun Sep 21 13:08:54 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.06             Driver Version: 570.124.06     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               Off |   00000000:81:00.0 Off |                  Off |
| 30%   29C    P8             18W /  230W |      18MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()


In [16]:

processor = AutoImageProcessor.from_pretrained('facebook/dinov2-with-registers-giant-imagenet1k-1-layer')

transform_test_Data = transforms.Compose([
     # Resize
        transforms.Resize(256),
        transforms.Lambda(lambda img: processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0))])


# Load the dataset using torchvision.datasets.ImageFolder and apply transformations
data_dir = "./FIT5215_Dataset"
full_dataset = datasets.ImageFolder(data_dir, transform = transform_test_Data)



# DataLoaders
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=False)#,num_workers=4, pin_memory=True)




    

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 7231.56it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
model = AutoModelForImageClassification.from_pretrained('facebook/dinov2-with-registers-giant-imagenet1k-1-layer')
model.classifier = nn.Linear(in_features=3072, out_features=20, bias=True)
model.load_state_dict(torch.load(f"./models/Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.pth"))
model.to(device)

Dinov2WithRegistersForImageClassification(
  (dinov2_with_registers): Dinov2WithRegistersModel(
    (embeddings): Dinov2WithRegistersEmbeddings(
      (patch_embeddings): Dinov2WithRegistersPatchEmbeddings(
        (projection): Conv2d(3, 1536, kernel_size=(14, 14), stride=(14, 14))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): Dinov2WithRegistersEncoder(
      (layer): ModuleList(
        (0-39): 40 x Dinov2WithRegistersLayer(
          (norm1): LayerNorm((1536,), eps=1e-06, elementwise_affine=True)
          (attention): Dinov2WithRegistersAttention(
            (attention): Dinov2WithRegistersSelfAttention(
              (query): Linear(in_features=1536, out_features=1536, bias=True)
              (key): Linear(in_features=1536, out_features=1536, bias=True)
              (value): Linear(in_features=1536, out_features=1536, bias=True)
            )
            (output): Dinov2WithRegistersSelfOutput(
              (dense): Linear(in_features=1536, out_f

In [6]:
!nvidia-smi


Sat Sep 20 03:06:00 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.06             Driver Version: 570.124.06     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               Off |   00000000:41:00.0 Off |                  Off |
| 30%   36C    P2             71W /  230W |    4731MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
torch.cuda.empty_cache()

In [5]:
from tqdm import tqdm
import pandas as pd

def compare_data(model, full_loader):
    total = 0
    df = { "ID": [],
               "Label Predicted": [],
       "Label Truth" : []
    }
    model.to(device)
    model.eval()
    with torch.no_grad():
        for inputs, labels in tqdm(full_loader,total = len(full_loader)):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            outputs_logits = outputs.logits
            predicted = torch.argmax(outputs_logits, dim=1)
            for pred ,label_true in zip(predicted,labels):
                df['ID'].append(total)
                df["Label Predicted"].append(class_names[pred.item()])
                df["Label Truth"].append(class_names[label_true.item()])
                total += 1
    df = pd.DataFrame(df)

    return df
                
        



In [27]:
df_606 = compare_data(model,full_loader)

100%|██████████| 296/296 [09:09<00:00,  1.86s/it]


In [28]:
df_606

,ID,Label Predicted,Label Truth
0,0,birds,birds
1,1,birds,birds
2,2,birds,birds
3,3,birds,birds
4,4,birds,birds
...,...,...,...
9461,9461,vases,vases
9462,9462,vases,vases
9463,9463,vases,vases
9464,9464,vases,vases


In [38]:
df_606_report = classification_report(df_606['Label Truth'] , df_606['Label Predicted'] , digits = 12,output_dict = True) 
df_606_report = pd.DataFrame(df_606_report).transpose()
df_606_report

,precision,recall,f1-score,support
birds,1.000000,1.000000,1.000000,512.000000
bottles,0.995381,0.997685,0.996532,432.000000
breads,1.000000,1.000000,1.000000,432.000000
butterflies,0.996016,1.000000,0.998004,500.000000
cakes,1.000000,0.997685,0.998841,432.000000
cats,1.000000,1.000000,1.000000,501.000000
chickens,1.000000,1.000000,1.000000,500.000000
cows,1.000000,1.000000,1.000000,500.000000
dogs,1.000000,1.000000,1.000000,501.000000
ducks,1.000000,1.000000,1.000000,496.000000


In [41]:
#model = AutoModelForImageClassification.from_pretrained('facebook/dinov2-with-registers-giant-imagenet1k-1-layer')
#model.classifier = nn.Linear(in_features=3072, out_features=20, bias=True)
#model.load_state_dict(torch.load(f"./models/Register_Dino_Random_Large_3.pth"))
#model.to(device)
#df_large_3 = compare_data(model,full_loader)
df_large_3_report = classification_report(df_large_3['Label Truth'] , df_large_3['Label Predicted'] , digits = 30,output_dict = True) 
df_large_3_report = pd.DataFrame(df_large_3_report).transpose()
df_large_3_report

,precision,recall,f1-score,support
birds,1.000000,1.000000,1.000000,512.000000
bottles,1.000000,1.000000,1.000000,432.000000
breads,0.997691,1.000000,0.998844,432.000000
butterflies,1.000000,1.000000,1.000000,500.000000
cakes,1.000000,0.997685,0.998841,432.000000
cats,1.000000,1.000000,1.000000,501.000000
chickens,1.000000,1.000000,1.000000,500.000000
cows,0.998004,1.000000,0.999001,500.000000
dogs,1.000000,1.000000,1.000000,501.000000
ducks,1.000000,1.000000,1.000000,496.000000


In [44]:
#model = AutoModelForImageClassification.from_pretrained('facebook/dinov2-with-registers-giant-imagenet1k-1-layer')
#model.classifier = nn.Linear(in_features=3072, out_features=20, bias=True)
#model.load_state_dict(torch.load(f"./models/Register_Dino_Random_Giant_seed77_train0.8_aug1_epoch20.pth"))
#model.to(device)
#df_large_77 = compare_data(model,full_loader)
#df_large_77_report = classification_report(df_large_77['Label Truth'] , df_large_77['Label Predicted'] , digits = 30,output_dict = True) 
#df_large_77_report = pd.DataFrame(df_large_77_report).transpose()
df_large_77_report.reset_index()

,index,precision,recall,f1-score,support
0,birds,1.000000,1.000000,1.000000,512.000000
1,bottles,1.000000,0.997685,0.998841,432.000000
2,breads,1.000000,1.000000,1.000000,432.000000
3,butterflies,1.000000,1.000000,1.000000,500.000000
4,cakes,1.000000,1.000000,1.000000,432.000000
5,cats,1.000000,1.000000,1.000000,501.000000
6,chickens,0.998004,1.000000,0.999001,500.000000
7,cows,0.997996,0.996000,0.996997,500.000000
8,dogs,1.000000,1.000000,1.000000,501.000000
9,ducks,1.000000,1.000000,1.000000,496.000000


In [45]:
model = AutoModelForImageClassification.from_pretrained('facebook/dinov2-with-registers-giant-imagenet1k-1-layer')
model.classifier = nn.Linear(in_features=3072, out_features=20, bias=True)
model.load_state_dict(torch.load(f"./models/Register_Dino_Random_Giant_seed6_train0.8_aug2_epoch25.pth"))
model.to(device)
df_giant_6 = compare_data(model,full_loader)
df_giant_6_report = classification_report(df_giant_6['Label Truth'] , df_giant_6['Label Predicted'] , digits = 30,output_dict = True) 
df_giant_6_report = pd.DataFrame(df_large_6_report).transpose()
df_giant_6_report.reset_index()

100%|██████████| 296/296 [09:15<00:00,  1.88s/it]


,index,precision,recall,f1-score,support
0,birds,1.000000,1.000000,1.000000,512.000000
1,bottles,0.997685,0.997685,0.997685,432.000000
2,breads,0.993103,1.000000,0.996540,432.000000
3,butterflies,0.994036,1.000000,0.997009,500.000000
4,cakes,1.000000,0.990741,0.995349,432.000000
5,cats,1.000000,1.000000,1.000000,501.000000
6,chickens,0.998000,0.998000,0.998000,500.000000
7,cows,0.997996,0.996000,0.996997,500.000000
8,dogs,1.000000,1.000000,1.000000,501.000000
9,ducks,0.997988,1.000000,0.998993,496.000000


In [7]:
#model = AutoModelForImageClassification.from_pretrained('facebook/dinov2-with-registers-giant-imagenet1k-1-layer')
#model.classifier = nn.Linear(in_features=3072, out_features=20, bias=True)
#model.load_state_dict(torch.load(f"./models/Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.pth"))
model.to(device)
df_beit_large = compare_data(model,full_loader)
df_beit_large_report = classification_report(df_beit_large['Label Truth'] , df_beit_large['Label Predicted'] , digits = 30,output_dict = True) 
df_beit_large_report = pd.DataFrame(df_beit_large_report).transpose()
df_beit_large_report.reset_index()

100%|██████████| 296/296 [08:38<00:00,  1.75s/it]


,index,precision,recall,f1-score,support
0,birds,1.000000,1.000000,1.000000,512.000000
1,bottles,0.995381,0.997685,0.996532,432.000000
2,breads,1.000000,1.000000,1.000000,432.000000
3,butterflies,0.996016,1.000000,0.998004,500.000000
4,cakes,1.000000,0.997685,0.998841,432.000000
5,cats,1.000000,1.000000,1.000000,501.000000
6,chickens,1.000000,1.000000,1.000000,500.000000
7,cows,1.000000,1.000000,1.000000,500.000000
8,dogs,1.000000,1.000000,1.000000,501.000000
9,ducks,1.000000,1.000000,1.000000,496.000000


In [19]:
model = AutoModelForImageClassification.from_pretrained('facebook/dinov2-with-registers-giant-imagenet1k-1-layer')
model.classifier = nn.Linear(in_features=3072, out_features=20, bias=True)
model.load_state_dict(torch.load(f"./models/Register_Dino_Random_Giant_seed67136_train0.7_aug1_epoch30.pth"))
model.to(device)
df_giant_7 = compare_data(model,full_loader)
df_giant_7_report = classification_report(df_giant_7['Label Truth'] , df_giant_7['Label Predicted'] , digits = 30,output_dict = True) 
df_giant_7_report = pd.DataFrame(df_giant_7_report).transpose()
df_giant_7_report.reset_index()

100%|██████████| 296/296 [08:33<00:00,  1.74s/it]


,index,precision,recall,f1-score,support
0,birds,1.000000,1.000000,1.000000,512.000000
1,bottles,1.000000,1.000000,1.000000,432.000000
2,breads,1.000000,1.000000,1.000000,432.000000
3,butterflies,1.000000,0.998000,0.998999,500.000000
4,cakes,1.000000,1.000000,1.000000,432.000000
5,cats,1.000000,1.000000,1.000000,501.000000
6,chickens,0.996008,0.998000,0.997003,500.000000
7,cows,1.000000,0.998000,0.998999,500.000000
8,dogs,1.000000,1.000000,1.000000,501.000000
9,ducks,0.997988,1.000000,0.998993,496.000000


# ENSEMBLE !


In [13]:
df_base = pd.read_csv('Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.csv')
df_base.head()

,ID,Label
0,0,dogs
1,1,handguns
2,2,seals
3,3,dogs
4,4,butterflies


In [33]:
df_second = pd.read_csv('Submission_Register_Dino_Random_Large_3.csv')
df_second = df_second[df_second['Label'].isin(['bottles','butterflies','seals','spiders','vases'])==True]
df_second.head()

,ID,Label
2,2,seals
4,4,butterflies
7,7,seals
10,10,spiders
18,18,spiders


In [34]:
df_third = pd.read_csv('Register_Dino_Random_Giant_seed77_train0.8_aug1_epoch20.csv')
df_third = df_third[df_third['Label'].isin(['cakes','elephants'])== True]
df_third.head()

,ID,Label
25,25,elephants
32,32,cakes
39,39,cakes
41,41,elephants
63,63,elephants


 ## Strategy
 1. If df_third predict cakes or elephants --> Predict Cakes or Elephant
 2. if df_second predicts ['bottles','butterflies','seals','spiders','vases'] -- Predict it
 3. The rest of the prediction will follow df_base


In [40]:
df_merge = df_base.merge(df_second, how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label_x':'Label Base' , 'Label_y':'Label Second'})
df_merge = df_merge.merge(df_third,how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label':'Label Third'})
df_merge
#df_merge[(df_merge['Label Base'] != df_merge['Label Third']) &(df_merge['Label Third'].notnull())]

,ID,Label Base,Label Second,Label Third
0,0,dogs,NaN,NaN
1,1,handguns,NaN,NaN
2,2,seals,seals,NaN
3,3,dogs,NaN,NaN
4,4,butterflies,butterflies,NaN
...,...,...,...,...
16162,16162,spiders,spiders,NaN
16163,16163,spiders,spiders,NaN
16164,16164,ducks,NaN,NaN
16165,16165,ducks,NaN,NaN


In [43]:
import numpy as np
df_merge['Final Label'] = np.where(df_merge['Label Third'].notnull() , df_merge['Label Third'] ,
                                   
                                   np.where(df_merge['Label Second'].notnull() , df_merge['Label Second'], df_merge['Label Base'])
                                  )
df_merge

,ID,Label Base,Label Second,Label Third,Final Label
0,0,dogs,NaN,NaN,dogs
1,1,handguns,NaN,NaN,handguns
2,2,seals,seals,NaN,seals
3,3,dogs,NaN,NaN,dogs
4,4,butterflies,butterflies,NaN,butterflies
...,...,...,...,...,...
16162,16162,spiders,spiders,NaN,spiders
16163,16163,spiders,spiders,NaN,spiders
16164,16164,ducks,NaN,NaN,ducks
16165,16165,ducks,NaN,NaN,ducks


In [46]:
df_final = df_merge[['ID','Final Label']]
df_final= df_final.rename(columns ={'Final Label':'Label'})
df_final.to_csv('Ensemble_AGAIN.csv', index=False)

In [47]:
df_final

,ID,Label
0,0,dogs
1,1,handguns
2,2,seals
3,3,dogs
4,4,butterflies
...,...,...
16162,16162,spiders
16163,16163,spiders
16164,16164,ducks
16165,16165,ducks


## Strategy 2 , Swap number 1 and 2

In [48]:
df_merge = df_base.merge(df_second, how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label_x':'Label Base' , 'Label_y':'Label Second'})
df_merge = df_merge.merge(df_third,how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label':'Label Third'})
df_merge
#df_merge[(df_merge['Label Base'] != df_merge['Label Third']) &(df_merge['Label Third'].notnull())]

,ID,Label Base,Label Second,Label Third
0,0,dogs,NaN,NaN
1,1,handguns,NaN,NaN
2,2,seals,seals,NaN
3,3,dogs,NaN,NaN
4,4,butterflies,butterflies,NaN
...,...,...,...,...
16162,16162,spiders,spiders,NaN
16163,16163,spiders,spiders,NaN
16164,16164,ducks,NaN,NaN
16165,16165,ducks,NaN,NaN


In [50]:
import numpy as np
df_merge['Final Label'] = np.where(df_merge['Label Second'].notnull() , df_merge['Label Second'] ,
                                   
                                   np.where(df_merge['Label Third'].notnull() , df_merge['Label Third'], df_merge['Label Base'])
                                  )
df_merge

,ID,Label Base,Label Second,Label Third,Final Label
0,0,dogs,NaN,NaN,dogs
1,1,handguns,NaN,NaN,handguns
2,2,seals,seals,NaN,seals
3,3,dogs,NaN,NaN,dogs
4,4,butterflies,butterflies,NaN,butterflies
...,...,...,...,...,...
16162,16162,spiders,spiders,NaN,spiders
16163,16163,spiders,spiders,NaN,spiders
16164,16164,ducks,NaN,NaN,ducks
16165,16165,ducks,NaN,NaN,ducks


In [51]:
df_final = df_merge[['ID','Final Label']]
df_final= df_final.rename(columns ={'Final Label':'Label'})
df_final.to_csv('Ensemble_AGAIN_v2.csv', index=False)

In [52]:
df_final

,ID,Label
0,0,dogs
1,1,handguns
2,2,seals
3,3,dogs
4,4,butterflies
...,...,...
16162,16162,spiders
16163,16163,spiders
16164,16164,ducks
16165,16165,ducks


# Strategy Chage Again (Butterfly to Third)

In [55]:
df_base = pd.read_csv('Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.csv')
#####
df_second = pd.read_csv('Submission_Register_Dino_Random_Large_3.csv')
df_second = df_second[df_second['Label'].isin(['bottles','seals','spiders','vases'])==True]
##
df_third = pd.read_csv('Register_Dino_Random_Giant_seed77_train0.8_aug1_epoch20.csv')
df_third = df_third[df_third['Label'].isin(['cakes','elephants','butterflies'])== True]


In [56]:
df_merge = df_base.merge(df_second, how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label_x':'Label Base' , 'Label_y':'Label Second'})
df_merge = df_merge.merge(df_third,how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label':'Label Third'})
df_merge
#df_merge[(df_merge['Label Base'] != df_merge['Label Third']) &(df_merge['Label Third'].notnull())]

,ID,Label Base,Label Second,Label Third
0,0,dogs,NaN,NaN
1,1,handguns,NaN,NaN
2,2,seals,seals,NaN
3,3,dogs,NaN,NaN
4,4,butterflies,NaN,butterflies
...,...,...,...,...
16162,16162,spiders,spiders,NaN
16163,16163,spiders,spiders,NaN
16164,16164,ducks,NaN,NaN
16165,16165,ducks,NaN,NaN


In [57]:
import numpy as np
df_merge['Final Label'] = np.where(df_merge['Label Second'].notnull() , df_merge['Label Second'] ,
                                   
                                   np.where(df_merge['Label Third'].notnull() , df_merge['Label Third'], df_merge['Label Base'])
                                  )
df_merge

,ID,Label Base,Label Second,Label Third,Final Label
0,0,dogs,NaN,NaN,dogs
1,1,handguns,NaN,NaN,handguns
2,2,seals,seals,NaN,seals
3,3,dogs,NaN,NaN,dogs
4,4,butterflies,NaN,butterflies,butterflies
...,...,...,...,...,...
16162,16162,spiders,spiders,NaN,spiders
16163,16163,spiders,spiders,NaN,spiders
16164,16164,ducks,NaN,NaN,ducks
16165,16165,ducks,NaN,NaN,ducks


In [58]:
df_final = df_merge[['ID','Final Label']]
df_final= df_final.rename(columns ={'Final Label':'Label'})
df_final.to_csv('Ensemble_AGAIN_v3.csv', index=False)

In [59]:
df_final 

,ID,Label
0,0,dogs
1,1,handguns
2,2,seals
3,3,dogs
4,4,butterflies
...,...,...
16162,16162,spiders
16163,16163,spiders
16164,16164,ducks
16165,16165,ducks


# Strategy Chage Again (Seal Spider to Third)

In [61]:
df_base = pd.read_csv('Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.csv')
#####
df_second = pd.read_csv('Submission_Register_Dino_Random_Large_3.csv')
df_second = df_second[df_second['Label'].isin(['bottles','vases'])==True]
##
df_third = pd.read_csv('Register_Dino_Random_Giant_seed77_train0.8_aug1_epoch20.csv')
df_third = df_third[df_third['Label'].isin(['cakes','elephants','butterflies','seals','spiders'])== True]


In [62]:
df_merge = df_base.merge(df_second, how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label_x':'Label Base' , 'Label_y':'Label Second'})
df_merge = df_merge.merge(df_third,how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label':'Label Third'})
df_merge
#df_merge[(df_merge['Label Base'] != df_merge['Label Third']) &(df_merge['Label Third'].notnull())]

,ID,Label Base,Label Second,Label Third
0,0,dogs,NaN,NaN
1,1,handguns,NaN,NaN
2,2,seals,NaN,seals
3,3,dogs,NaN,NaN
4,4,butterflies,NaN,butterflies
...,...,...,...,...
16162,16162,spiders,NaN,spiders
16163,16163,spiders,NaN,spiders
16164,16164,ducks,NaN,NaN
16165,16165,ducks,NaN,NaN


In [63]:
import numpy as np
df_merge['Final Label'] = np.where(df_merge['Label Second'].notnull() , df_merge['Label Second'] ,
                                   
                                   np.where(df_merge['Label Third'].notnull() , df_merge['Label Third'], df_merge['Label Base'])
                                  )
df_merge

,ID,Label Base,Label Second,Label Third,Final Label
0,0,dogs,NaN,NaN,dogs
1,1,handguns,NaN,NaN,handguns
2,2,seals,NaN,seals,seals
3,3,dogs,NaN,NaN,dogs
4,4,butterflies,NaN,butterflies,butterflies
...,...,...,...,...,...
16162,16162,spiders,NaN,spiders,spiders
16163,16163,spiders,NaN,spiders,spiders
16164,16164,ducks,NaN,NaN,ducks
16165,16165,ducks,NaN,NaN,ducks


In [64]:
df_final = df_merge[['ID','Final Label']]
df_final= df_final.rename(columns ={'Final Label':'Label'})
df_final.to_csv('Ensemble_AGAIN_v4.csv', index=False)
df_final

,ID,Label
0,0,dogs
1,1,handguns
2,2,seals
3,3,dogs
4,4,butterflies
...,...,...
16162,16162,spiders
16163,16163,spiders
16164,16164,ducks
16165,16165,ducks


## Step Back , Only Seal to Third

In [65]:
df_base = pd.read_csv('Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.csv')
#####
df_second = pd.read_csv('Submission_Register_Dino_Random_Large_3.csv')
df_second = df_second[df_second['Label'].isin(['bottles','vases','spiders'])==True]
##
df_third = pd.read_csv('Register_Dino_Random_Giant_seed77_train0.8_aug1_epoch20.csv')
df_third = df_third[df_third['Label'].isin(['cakes','elephants','butterflies','seals'])== True]


df_merge = df_base.merge(df_second, how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label_x':'Label Base' , 'Label_y':'Label Second'})
df_merge = df_merge.merge(df_third,how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label':'Label Third'})

import numpy as np
df_merge['Final Label'] = np.where(df_merge['Label Second'].notnull() , df_merge['Label Second'] ,
                                   
                                   np.where(df_merge['Label Third'].notnull() , df_merge['Label Third'], df_merge['Label Base'])
                                  )

df_final = df_merge[['ID','Final Label']]
df_final= df_final.rename(columns ={'Final Label':'Label'})
df_final.to_csv('Ensemble_AGAIN_v5.csv', index=False)
df_final

,ID,Label
0,0,dogs
1,1,handguns
2,2,seals
3,3,dogs
4,4,butterflies
...,...,...
16162,16162,spiders
16163,16163,spiders
16164,16164,ducks
16165,16165,ducks


## Remove Elephant From Third

In [66]:
df_base = pd.read_csv('Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.csv')
#####
df_second = pd.read_csv('Submission_Register_Dino_Random_Large_3.csv')
df_second = df_second[df_second['Label'].isin(['bottles','vases','spiders'])==True]
##
df_third = pd.read_csv('Register_Dino_Random_Giant_seed77_train0.8_aug1_epoch20.csv')
df_third = df_third[df_third['Label'].isin(['cakes','butterflies','seals'])== True]


df_merge = df_base.merge(df_second, how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label_x':'Label Base' , 'Label_y':'Label Second'})
df_merge = df_merge.merge(df_third,how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label':'Label Third'})

import numpy as np
df_merge['Final Label'] = np.where(df_merge['Label Second'].notnull() , df_merge['Label Second'] ,
                                   
                                   np.where(df_merge['Label Third'].notnull() , df_merge['Label Third'], df_merge['Label Base'])
                                  )

df_final = df_merge[['ID','Final Label']]
df_final= df_final.rename(columns ={'Final Label':'Label'})
df_final.to_csv('Ensemble_AGAIN_v6.csv', index=False)
df_final

,ID,Label
0,0,dogs
1,1,handguns
2,2,seals
3,3,dogs
4,4,butterflies
...,...,...
16162,16162,spiders
16163,16163,spiders
16164,16164,ducks
16165,16165,ducks


# Naive Ensemble

In [6]:
def majority_vote(row):
    labels = [row["Label_1"], row["Label_2"], row["Label_3"]]#, row["Label_4"], row["Label_5"]]
    counts = pd.Series(labels).value_counts()
    top_count = counts.max()
    winners = counts[counts == top_count].index.tolist()
    
    if len(winners) == 1:
        return winners[0]       # clear winner
    else:
        return row["Label_1"]   # tie-breaker

In [7]:
import pandas as pd

df1 = pd.read_csv('Register_Dino_Random_Giant_seed67136_train0.7_aug1_epoch30.csv')
df1 = df1.rename(columns={'Label':'Label_1'})

df2 = pd.read_csv('[old again] Register_Dino_Random_Giant_seed28467_train0.8_aug1_epoch30.csv')
df2 = df2.rename(columns={'Label':'Label_2'})

df3 = pd.read_csv('[old]Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.csv')
df3 = df3.rename(columns={'Label':'Label_3'})

df_combined = df1.merge(df2)
df_combined = df_combined.merge(df3)
df_combined["final_label"] = df_combined.apply(majority_vote, axis=1)
df_combined

,ID,Label_1,Label_2,Label_3,final_label
0,0,dogs,dogs,dogs,dogs
1,1,handguns,handguns,handguns,handguns
2,2,seals,seals,seals,seals
3,3,dogs,dogs,dogs,dogs
4,4,butterflies,butterflies,butterflies,butterflies
...,...,...,...,...,...
16162,16162,spiders,spiders,spiders,spiders
16163,16163,spiders,spiders,spiders,spiders
16164,16164,ducks,ducks,ducks,ducks
16165,16165,ducks,ducks,ducks,ducks


In [9]:
df_final = df_combined[['ID','final_label']]
df_final = df_final.rename(columns={'final_label':'Label'})
df_final.to_csv('Ensemble_Again_v7.csv', index=False)

# Ensemble Inception

In [10]:
import pandas as pd

df1 = pd.read_csv('Ensemble_AGAIN_v3.csv')
df1 = df1.rename(columns={'Label':'Label_1'})

df2 = pd.read_csv('Ensemble_AGAIN_v5.csv')
df2 = df2.rename(columns={'Label':'Label_2'})

df3 = pd.read_csv('Ensemble_Again_v7.csv')
df3 = df3.rename(columns={'Label':'Label_3'})

df_combined = df1.merge(df2)
df_combined = df_combined.merge(df3)
df_combined["final_label"] = df_combined.apply(majority_vote, axis=1)
df_combined

,ID,Label_1,Label_2,Label_3,final_label
0,0,dogs,dogs,dogs,dogs
1,1,handguns,handguns,handguns,handguns
2,2,seals,seals,seals,seals
3,3,dogs,dogs,dogs,dogs
4,4,butterflies,butterflies,butterflies,butterflies
...,...,...,...,...,...
16162,16162,spiders,spiders,spiders,spiders
16163,16163,spiders,spiders,spiders,spiders
16164,16164,ducks,ducks,ducks,ducks
16165,16165,ducks,ducks,ducks,ducks


In [11]:
df_combined[df_combined['Label_1']!=df_combined['final_label']]

,ID,Label_1,Label_2,Label_3,final_label


In [ ]:
df_final = df_combined[['ID','final_label']]
df_final = df_final.rename(columns={'final_label':'Label'})
df_final.to_csv('Ensemble_Again_v8.csv', index=False)

# Strategy One

In [20]:
df_base = pd.read_csv('Register_Dino_Random_Giant_seed67136_train0.7_aug1_epoch30.csv')
#####
df_second = pd.read_csv('[old]Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.csv')
df_second = df_second[df_second['Label'].isin(['chickens','cows','ducks','snakes'])==True]
##
df_third = pd.read_csv('[old]Register_Dino_Random_Giant_seed77_train0.8_aug1_epoch20.csv')
df_third = df_third[df_third['Label'].isin(['spiders','butterflies'])== True]


In [21]:
df_merge = df_base.merge(df_second, how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label_x':'Label Base' , 'Label_y':'Label Second'})
df_merge = df_merge.merge(df_third,how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label':'Label Third'})
df_merge
#df_merge[(df_merge['Label Base'] != df_merge['Label Third']) &(df_merge['Label Third'].notnull())]

,ID,Label Base,Label Second,Label Third
0,0,dogs,NaN,NaN
1,1,handguns,NaN,NaN
2,2,seals,NaN,NaN
3,3,dogs,NaN,NaN
4,4,butterflies,NaN,butterflies
...,...,...,...,...
16162,16162,spiders,NaN,spiders
16163,16163,spiders,NaN,spiders
16164,16164,ducks,ducks,NaN
16165,16165,ducks,ducks,NaN


In [22]:
import numpy as np
df_merge['Final Label'] = np.where(df_merge['Label Third'].notnull() , df_merge['Label Third'] ,
                                   
                                   np.where(df_merge['Label Second'].notnull() , df_merge['Label Second'], df_merge['Label Base'])
                                  )
df_merge

,ID,Label Base,Label Second,Label Third,Final Label
0,0,dogs,NaN,NaN,dogs
1,1,handguns,NaN,NaN,handguns
2,2,seals,NaN,NaN,seals
3,3,dogs,NaN,NaN,dogs
4,4,butterflies,NaN,butterflies,butterflies
...,...,...,...,...,...
16162,16162,spiders,NaN,spiders,spiders
16163,16163,spiders,NaN,spiders,spiders
16164,16164,ducks,ducks,NaN,ducks
16165,16165,ducks,ducks,NaN,ducks


In [25]:
df_final = df_merge[['ID','Final Label']]
df_final= df_final.rename(columns ={'Final Label':'Label'})
df_final.to_csv('Ensemble_AGAIN_v8.csv', index=False)

# Again

In [26]:
df_base = pd.read_csv('Register_Dino_Random_Giant_seed67136_train0.7_aug1_epoch30.csv')
#####
df_second = pd.read_csv('[old]Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.csv')
df_second = df_second[df_second['Label'].isin(['chickens','cows','ducks','snakes'])==True]
##
df_third = pd.read_csv('[old]Register_Dino_Random_Giant_seed77_train0.8_aug1_epoch20.csv')
df_third = df_third[df_third['Label'].isin(['spiders','butterflies'])== True]


In [27]:
df_merge = df_base.merge(df_second, how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label_x':'Label Base' , 'Label_y':'Label Second'})
df_merge = df_merge.merge(df_third,how = 'left',on = 'ID')
df_merge= df_merge.rename(columns ={'Label':'Label Third'})
df_merge
#df_merge[(df_merge['Label Base'] != df_merge['Label Third']) &(df_merge['Label Third'].notnull())]

,ID,Label Base,Label Second,Label Third
0,0,dogs,NaN,NaN
1,1,handguns,NaN,NaN
2,2,seals,NaN,NaN
3,3,dogs,NaN,NaN
4,4,butterflies,NaN,butterflies
...,...,...,...,...
16162,16162,spiders,NaN,spiders
16163,16163,spiders,NaN,spiders
16164,16164,ducks,ducks,NaN
16165,16165,ducks,ducks,NaN


In [28]:
import numpy as np
df_merge['Final Label'] = np.where(df_merge['Label Second'].notnull() , df_merge['Label Second'] ,
                                   
                                   np.where(df_merge['Label Third'].notnull() , df_merge['Label Third'], df_merge['Label Base'])
                                  )
df_merge

,ID,Label Base,Label Second,Label Third,Final Label
0,0,dogs,NaN,NaN,dogs
1,1,handguns,NaN,NaN,handguns
2,2,seals,NaN,NaN,seals
3,3,dogs,NaN,NaN,dogs
4,4,butterflies,NaN,butterflies,butterflies
...,...,...,...,...,...
16162,16162,spiders,NaN,spiders,spiders
16163,16163,spiders,NaN,spiders,spiders
16164,16164,ducks,ducks,NaN,ducks
16165,16165,ducks,ducks,NaN,ducks


In [29]:
df_final = df_merge[['ID','Final Label']]
df_final= df_final.rename(columns ={'Final Label':'Label'})
df_final.to_csv('Ensemble_AGAIN_v9.csv', index=False)

# Ensemble Inception

In [8]:
import pandas as pd

df1 = pd.read_csv('Ensemble_AGAIN_v5.csv')
df1 = df1.rename(columns={'Label':'Label_1'})

df2 = pd.read_csv('Ensemble_AGAIN_v3.csv')
df2 = df2.rename(columns={'Label':'Label_2'})

df3 = pd.read_csv('Ensemble_AGAIN_v4.csv')
df3 = df3.rename(columns={'Label':'Label_3'})

df_combined = df1.merge(df2)
df_combined = df_combined.merge(df3)
df_combined["final_label"] = df_combined.apply(majority_vote, axis=1)
df_combined

,ID,Label_1,Label_2,Label_3,final_label
0,0,dogs,dogs,dogs,dogs
1,1,handguns,handguns,handguns,handguns
2,2,seals,seals,seals,seals
3,3,dogs,dogs,dogs,dogs
4,4,butterflies,butterflies,butterflies,butterflies
...,...,...,...,...,...
16162,16162,spiders,spiders,spiders,spiders
16163,16163,spiders,spiders,spiders,spiders
16164,16164,ducks,ducks,ducks,ducks
16165,16165,ducks,ducks,ducks,ducks


In [9]:
df_combined[df_combined['Label_1']!=df_combined['final_label']]

,ID,Label_1,Label_2,Label_3,final_label


# All

In [7]:
def majority_vote(row):
    labels = [row["Label_1"], row["Label_2"], row["Label_3"], row["Label_4"], row["Label_5"], row["Label_6"], row["Label_7"], row["Label_8"], row["Label_9"], row["Label_10"], row["Label_11"], row["Label_12"], row["Label_13"]]
    counts = pd.Series(labels).value_counts()
    top_count = counts.max()
    winners = counts[counts == top_count].index.tolist()
    
    if len(winners) == 1:
        return winners[0]       # clear winner
    else:
        return row["Label_1"]   # tie-breaker

In [8]:
import pandas as pd

df1 = pd.read_csv('Ensemble_AGAIN_v5.csv')
df1 = df1.rename(columns={'Label':'Label_1'})

df2 = pd.read_csv('[old again] Register_Dino_Random_Giant_seed12924_train0.9_aug1_epoch20.csv')
df2 = df2.rename(columns={'Label':'Label_2'})

df3 = pd.read_csv('[old again] Register_Dino_Random_Giant_seed28467_train0.8_aug1_epoch30.csv')
df3 = df3.rename(columns={'Label':'Label_3'})

df4 = pd.read_csv('[old again] Register_Dino_Random_Giant_seed52263_train0.9_aug1_epoch35.csv')
df4 = df4.rename(columns={'Label':'Label_4'})

df5 = pd.read_csv('[old]Register_Dino_Random_Giant_seed6_train0.8_aug2_epoch25.csv')
df5 = df5.rename(columns={'Label':'Label_5'})

df6 = pd.read_csv('[old]Register_Dino_Random_Giant_seed77_train0.8_aug1_epoch20.csv')
df6 = df6.rename(columns={'Label':'Label_6'})

df7 = pd.read_csv('[old]Register_Dino_Random_Giant_seed97_train0.8_aug1_epoch25.csv')
df7 = df7.rename(columns={'Label':'Label_7'})

df8 = pd.read_csv('[old]Register_Dino_Random_Giant_seed99_train0.9_aug1_epoch23.csv')
df8 = df8.rename(columns={'Label':'Label_8'})

df9 = pd.read_csv('[old]Register_Dino_Random_Giant_seed505_train0.9_aug2_epoch30.csv')
df9 = df9.rename(columns={'Label':'Label_9'})

df10 = pd.read_csv('[old]Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.csv')
df10 = df10.rename(columns={'Label':'Label_10'})

df11 = pd.read_csv('[old]Register_Dino_Random_Giant_seed707_train0.8_aug1_epoch35.csv')
df11 = df11.rename(columns={'Label':'Label_11'})

df12 = pd.read_csv('[old]Register_Dino_Random_Giant_seed909_train0.8_aug2_epoch20.csv')
df12 = df12.rename(columns={'Label':'Label_12'})

df13 = pd.read_csv('[old]Register_Dino_Random_Giant_seed1010_train0.9_aug2_epoch25.csv')
df13 = df13.rename(columns={'Label':'Label_13'})

df_combined = df1.merge(df2)
df_combined = df_combined.merge(df3)
df_combined = df_combined.merge(df4)
df_combined = df_combined.merge(df5)
df_combined = df_combined.merge(df6)
df_combined = df_combined.merge(df7)
df_combined = df_combined.merge(df8)
df_combined = df_combined.merge(df9)
df_combined = df_combined.merge(df10)
df_combined = df_combined.merge(df11)
df_combined = df_combined.merge(df12)
df_combined = df_combined.merge(df13)
df_combined["final_label"] = df_combined.apply(majority_vote, axis=1)
df_combined

,ID,Label_1,Label_2,Label_3,Label_4,Label_5,Label_6,Label_7,Label_8,Label_9,Label_10,Label_11,Label_12,Label_13,final_label
0,0,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs
1,1,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns
2,2,seals,seals,seals,seals,seals,seals,seals,seals,seals,seals,seals,seals,seals,seals
3,3,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs
4,4,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16162,16162,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders
16163,16163,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders
16164,16164,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks
16165,16165,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks


In [9]:
df_combined[df_combined['Label_1']!=df_combined['final_label']]

,ID,Label_1,Label_2,Label_3,Label_4,Label_5,Label_6,Label_7,Label_8,Label_9,Label_10,Label_11,Label_12,Label_13,final_label
219,219,elephants,lions,lions,lions,lions,lions,lions,lions,lions,elephants,lions,seals,lions,lions
671,671,cakes,lipsticks,lipsticks,cakes,lipsticks,cakes,cakes,lipsticks,lipsticks,lipsticks,lipsticks,lipsticks,lipsticks,lipsticks
704,704,snakes,breads,breads,breads,breads,snakes,breads,snakes,breads,snakes,breads,breads,breads,breads
1302,1302,butterflies,fishes,fishes,fishes,butterflies,butterflies,fishes,fishes,butterflies,fishes,butterflies,fishes,butterflies,fishes
1556,1556,butterflies,snakes,snakes,snakes,snakes,butterflies,snakes,snakes,snakes,birds,snakes,birds,snakes,snakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14931,14931,vases,elephants,dogs,elephants,dogs,dogs,elephants,vases,dogs,vases,elephants,vases,elephants,elephants
15659,15659,bottles,cows,cows,lions,cows,bottles,cows,cows,bottles,bottles,cows,cows,cows,cows
15720,15720,elephants,elephants,lions,elephants,elephants,lions,elephants,lions,lions,elephants,lions,lions,lions,lions
16033,16033,bottles,vases,vases,bottles,vases,bottles,vases,vases,vases,vases,vases,vases,vases,vases


In [10]:
df_final = df_combined[['ID','final_label']]
df_final.to_csv('Ensemble_AGAIN_v11.csv', index=False)

In [28]:
def majority_vote_again(row):
    labels = [
        #row["Label_1"],
        row["Label_2"], row["Label_3"], row["Label_4"], row["Label_5"],
        row["Label_6"], row["Label_7"], row["Label_8"], row["Label_9"], row["Label_10"],
        row["Label_11"], row["Label_12"], row["Label_13"]
    ]
    
    counts = pd.Series(labels).value_counts()
    top_count = counts.max()
    winners = counts[counts == top_count].index.tolist()
    
    # If any label appears 6 or more times, pick that one
    if top_count >= 3:
        return counts.idxmax()
    
    # If there’s a single winner
    if len(winners) == 1:
        return winners[0]
    
    # Otherwise, default to Label_1
    return row["Label_2"]


In [29]:
import pandas as pd



df2 = pd.read_csv('[old again] Register_Dino_Random_Giant_seed12924_train0.9_aug1_epoch20.csv')
df2 = df2.rename(columns={'Label':'Label_2'})

df3 = pd.read_csv('[old again] Register_Dino_Random_Giant_seed28467_train0.8_aug1_epoch30.csv')
df3 = df3.rename(columns={'Label':'Label_3'})

df4 = pd.read_csv('[old again] Register_Dino_Random_Giant_seed52263_train0.9_aug1_epoch35.csv')
df4 = df4.rename(columns={'Label':'Label_4'})

df5 = pd.read_csv('[old]Register_Dino_Random_Giant_seed6_train0.8_aug2_epoch25.csv')
df5 = df5.rename(columns={'Label':'Label_5'})

df6 = pd.read_csv('[old]Register_Dino_Random_Giant_seed77_train0.8_aug1_epoch20.csv')
df6 = df6.rename(columns={'Label':'Label_6'})

df7 = pd.read_csv('[old]Register_Dino_Random_Giant_seed97_train0.8_aug1_epoch25.csv')
df7 = df7.rename(columns={'Label':'Label_7'})

df8 = pd.read_csv('[old]Register_Dino_Random_Giant_seed99_train0.9_aug1_epoch23.csv')
df8 = df8.rename(columns={'Label':'Label_8'})

df9 = pd.read_csv('[old]Register_Dino_Random_Giant_seed505_train0.9_aug2_epoch30.csv')
df9 = df9.rename(columns={'Label':'Label_9'})

df10 = pd.read_csv('[old]Register_Dino_Random_Giant_seed606_train0.8_aug1_epoch20.csv')
df10 = df10.rename(columns={'Label':'Label_10'})

df11 = pd.read_csv('[old]Register_Dino_Random_Giant_seed707_train0.8_aug1_epoch35.csv')
df11 = df11.rename(columns={'Label':'Label_11'})

df12 = pd.read_csv('[old]Register_Dino_Random_Giant_seed909_train0.8_aug2_epoch20.csv')
df12 = df12.rename(columns={'Label':'Label_12'})

df13 = pd.read_csv('[old]Register_Dino_Random_Giant_seed1010_train0.9_aug2_epoch25.csv')
df13 = df13.rename(columns={'Label':'Label_13'})

df_combined = df2.merge(df3)
#df_combined = df_combined.merge(df3)
df_combined = df_combined.merge(df4)
df_combined = df_combined.merge(df5)
df_combined = df_combined.merge(df6)
df_combined = df_combined.merge(df7)
df_combined = df_combined.merge(df8)
df_combined = df_combined.merge(df9)
df_combined = df_combined.merge(df10)
df_combined = df_combined.merge(df11)
df_combined = df_combined.merge(df12)
df_combined = df_combined.merge(df13)
df_combined["final_label"] = df_combined.apply(majority_vote_again, axis=1)
df_combined

,ID,Label_2,Label_3,Label_4,Label_5,Label_6,Label_7,Label_8,Label_9,Label_10,Label_11,Label_12,Label_13,final_label
0,0,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs
1,1,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns,handguns
2,2,seals,seals,seals,seals,seals,seals,seals,seals,seals,seals,seals,seals,seals
3,3,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs,dogs
4,4,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies,butterflies
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16162,16162,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders
16163,16163,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders,spiders
16164,16164,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks
16165,16165,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks,ducks


In [30]:
df_combined[df_combined['Label_2']!=df_combined['final_label']]

,ID,Label_2,Label_3,Label_4,Label_5,Label_6,Label_7,Label_8,Label_9,Label_10,Label_11,Label_12,Label_13,final_label
785,785,vases,vases,bottles,bottles,bottles,bottles,bottles,bottles,bottles,bottles,bottles,bottles,bottles
1102,1102,fishes,vases,vases,vases,vases,vases,vases,vases,vases,vases,vases,vases,vases
1167,1167,lions,elephants,cows,cows,cows,elephants,lions,cows,cows,elephants,cows,cows,cows
1379,1379,snakes,snakes,spiders,snakes,spiders,spiders,spiders,spiders,spiders,snakes,snakes,spiders,spiders
1786,1786,vases,bottles,cakes,bottles,bottles,bottles,vases,cakes,bottles,bottles,cakes,bottles,bottles
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15044,15044,dogs,chickens,chickens,chickens,chickens,chickens,chickens,chickens,chickens,chickens,cows,chickens,chickens
15720,15720,elephants,lions,elephants,elephants,lions,elephants,lions,lions,elephants,lions,lions,lions,lions
16038,16038,lions,bottles,bottles,bottles,bottles,bottles,lions,bottles,bottles,bottles,lions,bottles,bottles
16081,16081,lions,elephants,lions,elephants,elephants,elephants,elephants,lions,elephants,lions,elephants,elephants,elephants


In [31]:
df_final = df_combined[['ID','final_label']]
df_final.to_csv('Ensemble_AGAIN_v13.csv', index=False)